# 02. Model Training & Evaluation — Vehicle Casting Defect Detection

> **Runtime Environment:** Google Colab (GPU: T4 or better recommended)
> **Target Output:** `models/model.keras` (to download and place in local project)

This notebook trains deep learning models (Baseline Custom CNN & Transfer Learning with MobileNetV2) on the Kaggle Casting Product dataset, evaluates performance on holdout test data, and exports `model.keras` for local inference and Grad-CAM visualization.

## 1. Hardware & Environment Verification

In [ ]:
import tensorflow as tf
print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Detected: {gpus}")
else:
    print("⚠️ No GPU detected. In Colab, go to: Runtime -> Change runtime type -> Select T4 GPU.")

## 2. Dataset Setup in Google Colab

Run one of the options below to get the dataset into Colab:

In [ ]:
# OPTION A: Download directly via Kaggle API
# Upload your kaggle.json before running this cell
import os
if not os.path.exists("data/casting_data"):
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d ravirajsinh45/real-life-industrial-dataset-of-casting-product
    !unzip -q real-life-industrial-dataset-of-casting-product.zip -d data/
    print("Dataset unzipped successfully!")
else:
    print("Dataset directory already exists.")

## 3. Data Pipelines & Augmentation

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
DATA_DIR = "data/casting_data/casting_data"

# 85/15 train/val split
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "train"),
    validation_split=0.15,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="rgb",
    label_mode="binary",
    shuffle=True,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "train"),
    validation_split=0.15,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="rgb",
    label_mode="binary",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="rgb",
    label_mode="binary",
    shuffle=False,
)

class_names = train_ds.class_names
print(f"Class indices: {class_names}")

# Data augmentation pipeline
augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.10, fill_mode="reflect"),
    layers.RandomBrightness(0.15, value_range=(0, 255)),
    layers.RandomZoom((-0.10, 0.10), fill_mode="reflect"),
], name="augmentation_layers")

# Prefetch & cache
AUTOTUNE = tf.data.AUTOTUNE
train_ds_perf = train_ds.cache().shuffle(1000).map(
    lambda x, y: (augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

val_ds_perf = val_ds.cache().prefetch(AUTOTUNE)
test_ds_perf = test_ds.cache().prefetch(AUTOTUNE)

## 4. Build Model Architecture

In [ ]:
def build_model(use_transfer=True):
    inputs = keras.Input(shape=(224, 224, 3), name="input_image")
    
    if use_transfer:
        # MobileNetV2 pretrained backbone
        x = keras.applications.mobilenet_v2.preprocess_input(inputs)
        base_model = keras.applications.MobileNetV2(
            input_shape=(224, 224, 3),
            include_top=False,
            weights="imagenet"
        )
        base_model.trainable = False
        x = base_model(x, training=False)
    else:
        # Custom CNN
        x = layers.Rescaling(1./255)(inputs)
        x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Conv2D(128, (3, 3), activation="relu", padding="same", name="conv4_target")(x)
        x = layers.MaxPooling2D()(x)
        
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="prediction")(x)
    
    model = keras.Model(inputs, outputs, name="casting_defect_model")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall"), keras.metrics.AUC(name="auc")]
    )
    return model

model = build_model(use_transfer=True)
model.summary()

## 5. Training with Callbacks

In [ ]:
os.makedirs("models", exist_ok=True)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath="models/model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        verbose=1,
    ),
]

EPOCHS = 20
history = model.fit(
    train_ds_perf,
    validation_data=val_ds_perf,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## 6. Test Set Evaluation & Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Predict on holdout test set
y_true = []
y_probs = []
for imgs, lbls in test_ds_perf:
    y_probs.extend(model.predict(imgs, verbose=0).flatten())
    y_true.extend(lbls.numpy().flatten())

y_true = np.array(y_true)
y_probs = np.array(y_probs)
y_pred = (y_probs >= 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Test Set Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

## 7. Grad-CAM Verification & Export

In [ ]:
# Verify model file
print("Model saved at: models/model.keras")
print(f"File size: {os.path.getsize('models/model.keras') / (1024*1024):.2f} MB")

# Download to your local machine for the Streamlit app
try:
    from google.colab import files
    files.download('models/model.keras')
    print("Download initiated! Move this file into vehicle-defect-detection/models/model.keras")
except ImportError:
    print("Running outside Colab. Model is saved locally at models/model.keras.")